# Al-Mehdi: beginner walkthrough

This notebook uses only synthetic events. It shows how the 100 recommendation-only agents analyse an event, how the Human Authority Policy creates a consensus decision, and how the audit chain is verified. It does not connect to external systems or require an API key.

## 1. Import the project

Run Jupyter from the repository root. The small path check also supports opening this notebook from the `notebooks` folder.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

from al_mehdi.catalog import build_catalog
from al_mehdi.orchestrator import SafetyOrchestrator
from al_mehdi.simulator import build_scenario, scenario_names

## 2. Verify the council

The repository must contain exactly 100 unique agents split equally across 10 teams.

In [ ]:
agents = build_catalog()
team_counts = {}
for agent in agents:
    team_counts[agent.team] = team_counts.get(agent.team, 0) + 1

print('Agents:', len(agents))
print('Teams:', len(team_counts))
print(team_counts)
assert len(agents) == 100
assert all(count == 10 for count in team_counts.values())
assert all(agent.action_scope == 'recommend_only' for agent in agents)

## 3. Run a safe synthetic incident

In [ ]:
print('Available scenarios:', scenario_names())
orchestrator = SafetyOrchestrator(project_root / 'runtime' / 'notebook_demo.db')
event = build_scenario('prompt_injection')
report = orchestrator.analyze(event)
report.decision.to_dict()

## 4. Inspect the strongest findings

Each finding shows which signals were matched. No finding can execute an action.

In [ ]:
top_findings = sorted(report.findings, key=lambda item: item.risk_score, reverse=True)[:10]
[
    {
        'agent': finding.agent_name,
        'team': finding.team,
        'risk': finding.risk_score,
        'signals': finding.matched_signals,
    }
    for finding in top_findings
]

## 5. Verify the audit chain

In [ ]:
audit_result = orchestrator.audit.verify_chain()
audit_result
assert audit_result['valid'] is True

## What to learn next

1. Read `docs/ARCHITECTURE.md`.
2. Change only the synthetic event wording and compare team scores.
3. Run the automated test suite before changing agent rules.
4. Do not add real-system connectors until authentication, approvals and the threat model have been independently reviewed.